# Phase C: V1 rerun (fp32) + V2 stability + V3 confidence

**Two runtimes, in parallel:**
- **Runtime A, GPU (T4/L4):** run *Setup 1–5*, then **section A**.
- **Runtime B, CPU only** (Runtime → Change runtime type → CPU): run *Setup 1–5*, then **section B**.
- When both are done, run **section C** in either runtime.

Everything saves progress to Drive. **After a disconnect, run Setup again, then re-run the cell that was interrupted. It resumes and does not start over.** Every cell can be re-run safely.

## Setup 1: paths, Drive, version check

In [ ]:
from pathlib import Path
CODE_DIR = Path("/content/drive/MyDrive/csg520_code/src")
OUT_DIR  = Path("/content/drive/MyDrive/csg520_outputs")
CKPT_DIR = OUT_DIR / "checkpoints"
LOCAL    = Path("/content/leaf_xai")

from google.colab import drive
drive.mount("/content/drive")

# 1) the code on Drive must be the new version
for f in ("explain.py", "views.py", "robustness.py", "confidence.py"):
    assert (CODE_DIR / f).exists(), f"{f} is missing from {CODE_DIR}: upload it"
src = (CODE_DIR / "explain.py").read_text()
assert "def _to_frame" in src and "--check-accuracy" in src and "occ_logp" in src, \
    "explain.py on Drive is an OLD version: upload the latest one from your local src folder (42,092 bytes)"
print("code: new explain.py + views/robustness/confidence found")

# 2) the checkpoints and cached files the runs need
needed = ["checkpoints/a1_resnet50.pt", "checkpoints/a1_efficientnet_b0.pt", "checkpoints/a1_resnet50_scratch.pt",
          "checkpoints/a3_cnn.pt", "checkpoints/a4_vit.pt", "checkpoints/a2_rf.joblib",
          "features/a2_test_n400_s256.npz", "preds/a3_cnn_test_probs.npz", "preds/a4_vit_test_probs.npz",
          "split/splits.csv", "split/classes.json"]
missing = [f for f in needed if not (OUT_DIR / f).exists()]
for f in needed:
    print(f"  {f:<42}", "MISSING" if f in missing else "OK")
assert not missing, f"missing on Drive: {missing}"

Mounted at /content/drive
code: new explain.py + views/robustness/confidence found
  checkpoints/a1_resnet50.pt                 OK
  checkpoints/a1_efficientnet_b0.pt          OK
  checkpoints/a1_resnet50_scratch.pt         OK
  checkpoints/a3_cnn.pt                      OK
  checkpoints/a4_vit.pt                      OK
  checkpoints/a2_rf.joblib                   OK
  features/a2_test_n400_s256.npz             OK
  preds/a3_cnn_test_probs.npz                OK
  preds/a4_vit_test_probs.npz                OK
  split/splits.csv                           OK
  split/classes.json                         OK


## Setup 2: code to local disk, packages

In [ ]:
import shutil, os, subprocess, re, json
shutil.copytree(CODE_DIR, LOCAL / "src", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc"))
!pip -q install lime "timm>=0.9" kagglehub
import torch
HAS_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| CPU cores:", os.cpu_count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
GPU: none | CPU cores: 8


## Setup 3: PlantVillage colour images (models) + segmented variant (leaf masks only)

In [ ]:
import kagglehub
try:
    from google.colab import userdata
    os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
    os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
except Exception:
    pass
ds_path = Path(kagglehub.dataset_download("abdallahalidev/plantvillage-dataset"))

def find_variant(name):
    return next(p for p in sorted(ds_path.rglob(name))
                if p.is_dir() and sum(1 for c in p.iterdir() if c.is_dir()) == 38)

color, MASK_ROOT = find_variant("color"), find_variant("segmented")
raw = LOCAL / "data" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    raw.symlink_to(color)
print("colour :", color)
print("masks  :", MASK_ROOT)

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
colour : /kaggle/input/plantvillage-dataset/plantvillage dataset/color
masks  : /kaggle/input/plantvillage-dataset/plantvillage dataset/segmented


## Setup 4: frozen split

In [ ]:
import pandas as pd
for f in ("splits.csv", "classes.json"):
    shutil.copy(OUT_DIR / "split" / f, LOCAL / "data" / f)
counts = pd.read_csv(LOCAL / "data" / "splits.csv")["split"].value_counts().to_dict()
print(counts)
assert counts == {"train": 38013, "val": 8146, "test": 8146}

# Arguments shared by every command below. Data and split come from /content/leaf_xai/data (the defaults).
ARGS = f'--ckpt-dir "{CKPT_DIR}" --output-dir "{OUT_DIR}"'
%cd /content/leaf_xai/src

{'train': 38013, 'test': 8146, 'val': 8146}
/content/leaf_xai/src


## Setup 5: move the old fp16 V1 results out of the way (safe to re-run)
Any finished V1 arm without the new `occ_logp` metrics was made by the old code. It is moved to
`csg520_outputs/explain_fp16_firstpass/` (kept, not deleted), so the new runs start clean and later
**resume** after a disconnect instead of restarting. Arms produced by the new code, or still in progress, are left alone.

In [ ]:
exp, old = OUT_DIR / "explain", OUT_DIR / "explain_fp16_firstpass"
for arm_dir in sorted(exp.glob("a*")):
    s = arm_dir / "summary.json"
    if s.exists() and "occ_logp_lift_kappa" not in json.loads(s.read_text()).get("metrics", {}):
        old.mkdir(exist_ok=True)
        dest = old / arm_dir.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(arm_dir), str(dest))
        print("archived old result:", arm_dir.name)
print("current V1 folders:", [p.name for p in sorted(exp.glob("a*"))])

current V1 folders: ['a1_efficientnet_b0', 'a1_resnet50', 'a1_resnet50_scratch', 'a3_cnn', 'a4_vit']


# A. Runtime A (GPU): A1 arms, A3, A4

**A1. Smoke test** (3 images, throwaway folder). It checks that the new code runs on the GPU and writes the new metrics.

In [ ]:
assert HAS_GPU, "Section A needs a GPU runtime (Runtime → Change runtime type → T4/L4)"
!python explain.py --arms a3_cnn --limit 3 --fresh --ckpt-dir "{CKPT_DIR}" --output-dir /content/smoke --mask-root "{MASK_ROOT}"
m = json.loads(Path("/content/smoke/explain/a3_cnn/summary.json").read_text())["metrics"]
assert "occ_logp_lift_kappa" in m, "smoke run did not produce the new metrics: stop and check explain.py"
print("\nsmoke test OK")

**A2. V1 rerun, fp32.** Five arms × 380 images, about 5–15 min per arm. Resumes after a disconnect.

In [ ]:
assert HAS_GPU
!python explain.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit {ARGS} --mask-root "{MASK_ROOT}" 2>&1 | tee -a "{OUT_DIR}/explain_fp32_gpu_log.txt"

**A3. V3 confidence.** A3 and A4 read their saved probabilities. The three A1 arms are scored once on the full test set (8,146 images) and saved to `preds/`.
Check that the macro-F1 in each printout matches the `metrics/*.json` value next to it.

In [ ]:
assert HAS_GPU
!python confidence.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit {ARGS} 2>&1 | tee -a "{OUT_DIR}/confidence_gpu_log.txt"

**A4. V2 stability.** 760 test images × 16 conditions per arm. Takes a few minutes per arm and resumes after a disconnect.

In [ ]:
assert HAS_GPU
!python robustness.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit {ARGS} 2>&1 | tee -a "{OUT_DIR}/robustness_gpu_log.txt"

# B. Runtime B (CPU): A2 random forest
The RF file is about 1.2 GB, so each step spends about a minute loading it.

**B1. V3 confidence for A2**, from the cached test features (the exact rows behind the reported 0.881).

In [ ]:
!python confidence.py --arms a2_rf --n-jobs -1 {ARGS} 2>&1 | tee -a "{OUT_DIR}/confidence_cpu_log.txt"


=== V3 / a2_rf ===  n=6953  macro-F1=0.8811 (metrics/a2_rf.json: 0.8811)
  confidence  correct 0.672  wrong 0.319  gap 0.353 [0.341, 0.363]  (n wrong = 754)
  mistakes > 0.9: 2/754  AUROC 0.891  ECE 0.258  NLL 0.638

                arm subset    n  accuracy  n_wrong  conf_correct_mean  conf_wrong_mean   gap  wrong_above_0.9_n  wrong_above_0.9_share  auroc_right_vs_wrong   ece
 a1_efficientnet_b0 common 6953     0.998       17              0.888            0.597 0.291                  3                  0.176                 0.830 0.110
        a1_resnet50 common 6953     0.997       20              0.892            0.633 0.259                  3                  0.150                 0.834 0.106
a1_resnet50_scratch common 6953     0.986       95              0.902            0.559 0.343                  4                  0.042                 0.941 0.089
              a2_rf common 6953     0.892      754              0.672            0.319 0.353                  2                  0

**B2. Accuracy check before the long A2 V1 run.** This computes no maps. It measures A2's clean accuracy on the 380 V1 images through the explain path, which should come out near **0.89**. It also prints diagnostics: the bare-crop accuracy, whether the re-extracted features match the cached training features, the RF accuracy on those cached rows, and whether PIL and cv2 decode identically. If the check fails, the cell stops, B3 will not run, and you should send the printed diagnostics.

In [ ]:
cmd = ["python", "explain.py", "--arms", "a2_rf", "--check-accuracy", "--n-jobs", "-1",
       "--ckpt-dir", str(CKPT_DIR), "--output-dir", str(OUT_DIR)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-4000:], res.stderr[-4000:])
found = re.search(r"clean accuracy on \d+ sample images: ([0-9.]+)", res.stdout)
assert found, "accuracy line not found: see the error above"
A2_ACC = float(found.group(1))
A2_OK = A2_ACC >= 0.80
assert A2_OK, f"A2 sample accuracy {A2_ACC:.3f} is too low (expected ~0.89): do not run B3; send the diagnostics above"
print(f"A2 accuracy {A2_ACC:.3f}: OK, run B3")

[explain] sample: 380 images (10/class), LIME on 114 (A2: 2/class) | fill=mean mean RGB=(110.5, 120.0, 93.1)
[explain] loading a2_rf.joblib (the RF is large; this can take a minute)
[a2_rf] clean accuracy on 380 sample images: 0.892  (A2 test accuracy is 0.89)
[a2_rf] diagnostics:
  bare 224 crop (old behaviour)       : 0.682
  PIL vs cv2 decode, max pixel diff   : median 0, max 0  (0 = identical)
  sample images also in the A2 cache  : 359
  RF accuracy on CACHED features      : 0.889
  RF accuracy on RE-EXTRACTED features: 0.889
  feature max |diff| per image        : median 0.00e+00, max 0.00e+00  (~0 = features reproduce)
  largest differing features          : color_hist_h_00=0, color_hist_h_01=0, color_hist_h_02=0, color_hist_h_03=0, color_hist_h_04=0
 
A2 accuracy 0.892: OK, run B3


**B3. A2 V1 run (occlusion + LIME, full-frame features).** Long: 30–90 min depending on cores. Resumes after a disconnect (re-run B2 first; it is quick).

In [ ]:
assert globals().get("A2_OK"), "run B2 first (and it must pass)"
!python explain.py --arms a2_rf --n-jobs -1 {ARGS} --mask-root "{MASK_ROOT}" 2>&1 | tee -a "{OUT_DIR}/explain_fp32_a2_log.txt"

**B4. V2 stability for A2.** 760 images × 16 conditions, all feature extraction on CPU, roughly 30–60 min. Resumes after a disconnect.

In [ ]:
!python robustness.py --arms a2_rf --n-jobs -1 {ARGS} 2>&1 | tee -a "{OUT_DIR}/robustness_cpu_log.txt"

[robustness] 760 test images (20/class), 15 perturbed conditions + clean
[explain] loading a2_rf.joblib (the RF is large; this can take a minute)
[a2_rf] clean            acc=0.893  (10s)
[a2_rf] brightness_0.6   acc=0.517  (4s)
[a2_rf] brightness_0.8   acc=0.743  (5s)
[a2_rf] brightness_1.2   acc=0.778  (4s)
[a2_rf] brightness_1.4   acc=0.638  (5s)
[a2_rf] contrast_0.6     acc=0.391  (5s)
[a2_rf] contrast_0.8     acc=0.639  (4s)
[a2_rf] contrast_1.2     acc=0.755  (4s)
[a2_rf] contrast_1.4     acc=0.582  (5s)
[a2_rf] rotation_-30     acc=0.750  (4s)
[a2_rf] rotation_-15     acc=0.772  (4s)
[a2_rf] rotation_15      acc=0.780  (4s)
[a2_rf] rotation_30      acc=0.762  (4s)
[a2_rf] blur_1           acc=0.368  (4s)
[a2_rf] blur_2           acc=0.316  (4s)
[a2_rf] blur_3           acc=0.278  (4s)

=== V2 / a2_rf ===  images=760  clean acc=0.893
  brightness  flip 0.301 [0.281, 0.322]  retention 0.749  conf drop +0.239  worst level 0.463
  contrast    flip 0.375 [0.354, 0.396]  retention 0.6

In [ ]:
!python explain.py --arms a2_rf --methods occlusion --fill blur --n-jobs -1 {ARGS} --output-dir "{OUT_DIR}/sens_blurfill" --mask-root "{MASK_ROOT}" 2>&1 | tee -a "{OUT_DIR}/sens_blurfill_a2_log.txt"

[explain] sample: 380 images (10/class), LIME on 114 (A2: 2/class) | fill=blur mean RGB=(110.5, 120.0, 93.1)
[explain] mask index: 54306 segmented images
[explain] loading a2_rf.joblib (the RF is large; this can take a minute)
[a2_rf] 10/380 images  (0.7s/img, ~4 min left)
[a2_rf] 20/380 images  (0.7s/img, ~4 min left)
[a2_rf] 30/380 images  (0.7s/img, ~4 min left)
[a2_rf] 40/380 images  (0.7s/img, ~4 min left)
[a2_rf] 50/380 images  (0.6s/img, ~4 min left)
[a2_rf] 60/380 images  (0.6s/img, ~3 min left)
[a2_rf] 70/380 images  (0.6s/img, ~3 min left)
[a2_rf] 80/380 images  (0.6s/img, ~3 min left)
[a2_rf] 90/380 images  (0.6s/img, ~3 min left)
[a2_rf] 100/380 images  (0.6s/img, ~3 min left)
[a2_rf] 110/380 images  (0.6s/img, ~3 min left)
[a2_rf] 120/380 images  (0.6s/img, ~3 min left)
[a2_rf] 130/380 images  (0.6s/img, ~3 min left)
[a2_rf] 140/380 images  (0.6s/img, ~3 min left)
[a2_rf] 150/380 images  (0.6s/img, ~2 min left)
[a2_rf] 160/380 images  (0.6s/img, ~2 min left)
[a2_rf] 170/38

# C. Tables and figures (either runtime, once A and B are done)

In [ ]:
!python explain.py --summarize {ARGS}
!python robustness.py --summarize {ARGS}
!python confidence.py --summarize {ARGS}
from IPython.display import display
for name, path in [("V1", OUT_DIR / "explain" / "v1_table.csv"),
                   ("V2", OUT_DIR / "robustness" / "v2_table.csv"),
                   ("V3", OUT_DIR / "confidence" / "v3_table.csv")]:
    if path.exists():
        print(f"\n==== {name}: {path.name}"); display(pd.read_csv(path))
    else:
        print(f"\n==== {name}: not produced yet")

                arm  sample_acc  null_leaf_fraction  occ_signal_share  occ_lift_kappa  occ_lift_kappa_signal_only  occ_logp_lift_kappa  lime_lift_kappa  spearman_occ_lime
 a1_efficientnet_b0       0.997               0.587             0.024           0.109                       0.353                0.110            0.192              0.010
        a1_resnet50       0.997               0.587             0.042           0.074                       0.389                0.075            0.403             -0.001
a1_resnet50_scratch       0.982               0.587             0.334           0.211                       0.393                0.215            0.006              0.133
              a2_rf       0.892               0.587             0.829           0.097                       0.087                0.070            0.223              0.158
             a3_cnn       0.997               0.587             0.266           0.155                       0.296                0.157           

,arm,n_images,sample_acc,null_leaf_fraction,occ_score,occ_score_ci,occ_lift_ratio,occ_lift_ratio_ci,occ_lift_kappa,occ_lift_kappa_ci,...,lime_lift_kappa_ci,spearman_occ_lime,spearman_occ_lime_ci,lime_r2,lime_r2_ci,occ_lift_kappa_signal_only,occ_lift_kappa_signal_only_ci,occ_signal_share,occ_share_above_chance,lime_share_above_chance
0,a1_efficientnet_b0,380,0.997368,0.586579,0.648508,"[0.631, 0.666]",1.171942,"[1.134, 1.215]",0.109425,"[0.065, 0.152]",...,"[0.127, 0.253]",0.009914,"[-0.031, 0.051]",0.615743,"[0.589, 0.641]",0.352651,"[0.229, 0.473]",0.023684,0.663158,0.745614
1,a1_resnet50,380,0.997368,0.586579,0.615888,"[0.596, 0.636]",1.062702,"[1.035, 1.092]",0.073944,"[0.039, 0.108]",...,"[0.351, 0.449]",-0.001123,"[-0.041, 0.040]",0.655947,"[0.633, 0.677]",0.389002,"[0.270, 0.514]",0.042105,0.615789,0.894737
2,a1_resnet50_scratch,380,0.981579,0.586579,0.685189,"[0.661, 0.709]",1.245301,"[1.189, 1.301]",0.210797,"[0.150, 0.269]",...,"[-0.071, 0.077]",0.132898,"[0.096, 0.169]",0.688220,"[0.659, 0.714]",0.393208,"[0.305, 0.475]",0.334211,0.660526,0.570175
3,a2_rf,380,0.892105,0.586579,0.647619,"[0.622, 0.673]",1.162045,"[1.110, 1.218]",0.096660,"[0.021, 0.169]",...,"[0.145, 0.300]",0.158002,"[0.101, 0.212]",0.649015,"[0.615, 0.686]",0.087055,"[0.007, 0.161]",0.828947,0.611111,0.763158
4,a3_cnn,380,0.997368,0.586579,0.655423,"[0.630, 0.679]",1.150383,"[1.105, 1.196]",0.154578,"[0.090, 0.213]",...,"[0.134, 0.248]",0.147495,"[0.100, 0.196]",0.759076,"[0.731, 0.783]",0.295784,"[0.207, 0.377]",0.265789,0.656085,0.771930
5,a4_vit,380,0.997368,0.586579,0.679767,"[0.660, 0.699]",1.228659,"[1.188, 1.276]",0.205469,"[0.164, 0.248]",...,"[0.148, 0.276]",0.117412,"[0.075, 0.160]",0.722455,"[0.705, 0.740]",0.428447,"[0.304, 0.555]",0.047368,0.717678,0.754386



==== V2: v2_table.csv


,arm,n_images,clean_acc,brightness_flip,brightness_flip_ci,brightness_retention,contrast_flip,contrast_flip_ci,contrast_retention,rotation_flip,rotation_flip_ci,rotation_retention,blur_flip,blur_flip_ci,blur_retention,overall_flip,overall_flip_ci,overall_retention
0,a1_efficientnet_b0,760,0.998684,0.001316,"[0.000, 0.003]",0.998682,0.001316,"[0.000, 0.003]",0.998682,0.000329,"[0.000, 0.001]",0.999671,0.133772,"[0.120, 0.150]",0.866052,0.027544,"[0.024, 0.031]",0.972420
1,a1_resnet50,760,0.997368,0.000987,"[0.000, 0.003]",1.000000,0.000987,"[0.000, 0.003]",1.000000,0.002632,"[0.000, 0.006]",0.999670,0.216667,"[0.198, 0.236]",0.785400,0.044561,"[0.040, 0.049]",0.956992
2,a1_resnet50_scratch,760,0.985526,0.013158,"[0.009, 0.018]",0.993324,0.021053,"[0.015, 0.028]",0.987984,0.011184,"[0.006, 0.017]",0.996662,0.348246,"[0.324, 0.375]",0.658656,0.081754,"[0.075, 0.089]",0.925857
3,a2_rf,760,0.893421,0.300987,"[0.281, 0.322]",0.748895,0.375329,"[0.354, 0.396]",0.662371,0.199013,"[0.173, 0.226]",0.857511,0.670175,"[0.640, 0.701]",0.358861,0.367456,"[0.350, 0.386]",0.676780
4,a3_cnn,760,0.998684,0.002961,"[0.001, 0.006]",0.997036,0.003289,"[0.001, 0.006]",0.996706,0.000987,"[0.000, 0.002]",0.999012,0.402193,"[0.382, 0.425]",0.598155,0.082368,"[0.078, 0.087]",0.917699
5,a4_vit,760,0.998684,0.000000,"[0.000, 0.000]",1.000000,0.000329,"[0.000, 0.001]",0.999671,0.000329,"[0.000, 0.001]",0.999671,0.048684,"[0.039, 0.059]",0.951252,0.009912,"[0.008, 0.012]",0.990075



==== V3: v3_table.csv


,arm,subset,n,accuracy,macro_f1,n_wrong,conf_correct_mean,conf_correct_median,conf_wrong_mean,conf_wrong_median,...,wrong_above_0.9_n,wrong_above_0.9_share,wrong_above_0.9_ci95,correct_above_0.9_share,auroc_right_vs_wrong,ece,mce,nll,brier,max_conf_seen
0,a1_efficientnet_b0,common,6953,0.997555,0.996410,17,0.888379,0.914474,0.597428,0.551725,...,3,0.176471,"[0.062, 0.410]",0.606978,0.829695,0.109887,0.634611,0.129034,0.023398,0.999779
1,a1_resnet50,common,6953,0.997124,0.996372,20,0.892354,0.919102,0.633013,0.637531,...,3,0.150000,"[0.052, 0.360]",0.644454,0.833838,0.105685,0.358133,0.124851,0.022648,0.998130
2,a1_resnet50_scratch,common,6953,0.986337,0.981721,95,0.902078,0.941784,0.559052,0.531417,...,4,0.042105,"[0.016, 0.103]",0.666375,0.940974,0.089293,0.318136,0.133035,0.036003,0.999885
3,a2_rf,common,6953,0.891558,0.881088,754,0.671906,0.698000,0.318923,0.290000,...,2,0.002653,"[0.001, 0.010]",0.217616,0.890993,0.257930,0.426880,0.637825,0.258673,1.000000
4,a3_cnn,common,6953,0.995541,0.994208,31,0.901991,0.939806,0.600769,0.570386,...,3,0.096774,"[0.033, 0.249]",0.684051,0.903505,0.094894,0.755728,0.118101,0.025224,0.999861
5,a4_vit,common,6953,0.998274,0.997504,12,0.881087,0.906478,0.598593,0.563354,...,2,0.166667,"[0.047, 0.448]",0.556548,0.857477,0.117878,0.863312,0.134982,0.023444,0.994772
6,a1_efficientnet_b0,full,8146,0.997790,0.996349,18,0.866652,0.903299,0.602159,0.555166,...,3,0.166667,"[0.058, 0.392]",0.517963,0.814373,0.131722,0.634611,0.154806,0.030701,0.999779
7,a1_resnet50,full,8146,0.997422,0.996279,21,0.870511,0.908700,0.609066,0.599357,...,3,0.142857,"[0.050, 0.346]",0.549908,0.809090,0.127762,0.354326,0.150701,0.029759,0.998130
8,a1_resnet50_scratch,full,8146,0.986619,0.979218,109,0.883648,0.925802,0.542498,0.512457,...,4,0.036697,"[0.014, 0.091]",0.569491,0.934155,0.107686,0.316048,0.153665,0.040868,0.999885
9,a2_rf,full,6953,0.891558,0.881088,754,0.671906,0.698000,0.318923,0.290000,...,2,0.002653,"[0.001, 0.010]",0.217616,0.890993,0.257930,0.426880,0.637825,0.258673,1.000000


In [ ]:
from IPython.display import Image
for p in [OUT_DIR / "robustness" / "v2_flip_curves.png", OUT_DIR / "confidence" / "v3_confidence.png",
          *sorted((OUT_DIR / "explain").glob("a*/examples.png"))]:
    if p.exists():
        print(p.relative_to(OUT_DIR)); display(Image(str(p)))